# Beyond First Attendance: Institutional Incentives, High-Frequency Mathematics Support Use, and Heterogeneous Academic Outcomes

**Paper 2.1 — executable notebook version**

This notebook mirrors the scientific argument of the Paper 2.1 manuscript while exposing the code that produces the reported quantities. It is intended to be read from top to bottom as a computational paper: institutional context and hypotheses are presented in Markdown; the corresponding cohort construction, estimators, tables, and diagnostics appear immediately afterward in executable Python cells.

The notebook intentionally reuses the same `cmat_analysis` API and the same Paper 2.1 grouping rules as the manuscript. It does **not** maintain a separate statistical implementation.

> **Interpretive rule.** CMAT attendance is observational. The notebook describes outcome differences associated with recorded attendance and visit frequency; it does not identify a causal dose-response effect of additional visits.


## Abstract

Mathematics-support attendance is often summarized as a binary indicator, although repeated use may reflect different mixtures of academic need, voluntary help-seeking, and institutional encouragement. We analyze 6,627 first eligible attempts at a first-year university mathematics course, linking support-centre visits to the same academic period and standardizing final performance within instructor-period groups.

The main reporting grid distinguishes 0, exact 1–6, and 7+ visits. During at least part of the study period, three CMAT visits could satisfy a first-semester PPA participation activity, so the first three visits cannot be interpreted as a purely voluntary dose of support. The paper therefore separates the large **zero-versus-any-attendance margin** from the structure of repeated use among students who attend.

The current results show a strong adjusted 0-versus-1+ association in both standardized performance and pass probability, but no monotone visit-by-visit staircase among users. The 7+ group has the highest observed pass rate and is the clearest high-frequency profile. Distributional analyses further show that administrative outcomes, especially BV, materially shape the lower tail of the completed continuous outcome. These patterns remain observational and should not be interpreted as causal effects of CMAT attendance.


## Introduction and literature framing

Mathematics and statistics support has become a routine component of higher education, particularly where students enter quantitatively demanding programmes with heterogeneous mathematical preparation. The literature nevertheless makes evaluation difficult because attendance is behavioral rather than assigned: students may attend because they are struggling, because they are motivated to improve, because they already value academic support, or because institutional structures encourage participation.

Much of the observational mathematics-support literature therefore concentrates on a first margin—**users versus non-users**—and frequently reports more favorable outcomes among students who attend. At the same time, identification-oriented work using randomized encouragement, instrumental variables, or stronger quasi-experimental adjustment has produced more mixed evidence about direct academic effects. Paper 2.1 treats this distinction as central: a large user/non-user association can be institutionally important without showing what would happen if an otherwise identical student were assigned an additional visit.

Repeated use introduces a second margin. Prior work, including the longitudinal mathematics-support literature cited in the manuscript, has separated one-time engagement from higher-frequency attendance. In the present setting this distinction is especially important because the first three CMAT visits may have been partly influenced by PPA participation credit. Visit frequency is therefore not modeled as a homogeneous treatment dose.

A third issue is distributional. Average grades can conceal a lower tail containing numeric failures and administrative course endings. For this reason, the paper studies both observed numeric grades and the completed continuous outcome after the pre-specified numerical treatment of BV, RT, and BA, and it compares two-Gaussian mixtures with a flexible one-component skew-normal alternative rather than equating non-normality with latent classes.

The notebook follows the manuscript's evidentiary hierarchy: broad engagement margin first, frequency among users second, observed adverse-outcome composition third, and distribution-shape diagnostics last. The full bibliography remains in `paper/references.bib`, which is displayed at the end.


## 1. Reproducibility contract

The canonical analysis has three layers:

1. `code/run_paper21.py` — outcome-blind support audit used to choose a defensible upper-tail grouping.
2. `code/run_paper21_outcomes.py` — benchmark, frequency, PASS, outcome-composition, and robustness analyses.
3. `code/run_paper21_mixture.py` — numeric-failure, Gaussian-mixture, skew-normal, bootstrap, and cross-validation analyses.

The notebook reproduces the central light-to-moderate computations directly with the same reusable functions. Expensive mixture bootstrap/CV calculations are optional because the verified aggregate outputs are already versioned under `results/paper21/tables/`.


### Two reading layers

This notebook is intentionally written for two audiences. The main section prose follows the substantive interpretation intended for readers and institutional stakeholders. Immediately before each code block, **What this block does** explains the computational role of the cell, while **What is happening mathematically / technically?** documents the exact estimator, parameterization, transformations, hypothesis tests, covariance construction, or numerical algorithm for a statistical reader.

The technical layer is meant to be auditable against the implementation: when the code uses OLS, GLM-logit, empirical standardization, delta-method uncertainty, EM, parametric bootstrap, or repeated cross-validation, those operations are stated explicitly rather than summarized as an unspecified “adjusted model.”


### 1.1 Reproducible execution environment

#### What this block does

This block makes the notebook portable inside the repository. It locates the repository root, defines the canonical `paper/`, `results/paper21/tables/`, and `results/paper21/figures/` paths, exposes the local `cmat_analysis` package to Python, and sets display options.

#### What is happening mathematically / technically?

There is no statistical estimation in this block. The relevant technical point is **path and implementation identity**: every later object is resolved relative to one detected repository root, so the notebook does not silently switch to another dataset, another copy of the library, or a user-specific absolute path.

`find_repo_root()` walks upward from the current working directory until it finds `paper/visit_grouping_spec.json`. The local source tree is then prepended to `sys.path`, which means imports below resolve to the version of `cmat_analysis` checked out in this branch. Consequently, the notebook and the paper runners execute the same source code rather than two independently installed implementations.


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "paper" / "visit_grouping_spec.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the CMAT-research root. Run this notebook from inside the repository."
    )

REPO_ROOT = find_repo_root(Path.cwd())
PAPER_DIR = REPO_ROOT / "paper"
TABLES_DIR = REPO_ROOT / "results" / "paper21" / "tables"
FIGURES_DIR = REPO_ROOT / "results" / "paper21" / "figures"

# Make the shared editable library importable even when it has not been pip-installed.
CMAT_SRC = REPO_ROOT / "cmat_analysis" / "src"
if str(CMAT_SRC) not in sys.path:
    sys.path.insert(0, str(CMAT_SRC))

print("Repository:", REPO_ROOT)
print("Paper 2.1 tables:", TABLES_DIR)
print("Paper 2.1 figures:", FIGURES_DIR)



### 1.2 Canonical statistical API and fixed analysis order

#### What this block does

This block imports the cohort constructors, outcome constructors, regression estimators, mixture-model estimators, and plotting code used throughout Paper 2.1. It also fixes the reporting order `0,1,2,3,4,5,6,7+` and the five academic-result states.

#### What is happening mathematically / technically?

The important design decision is that the notebook contains **orchestration**, not duplicate estimators. For example, `fixed_effect_group_comparisons()` is the same function called by `code/run_paper21_outcomes.py`, and the GMM functions are the same implementations used by `code/run_paper21_mixture.py`.

The ordered group vectors are not cosmetic. They determine the treatment-reference coding used in the regression design matrix. When

$$
\texttt{GROUP\_ORDER}=(0,1,2,3,4,5,6,7+),
$$

the first level, $0$, becomes the reference category in the zero-inclusive models, so the corresponding group coefficients are parameterized relative to zero attendance. Pairwise contrasts between non-reference groups are then obtained as linear combinations of those fitted coefficients.


In [ ]:
from cmat_analysis.cohorts import build_study_cohorts, load_and_clean_inputs
from cmat_analysis.config.study_config import get_study_config
from cmat_analysis.measures import add_academic_outcome_states, add_primary_outcomes
from cmat_analysis.statistics import (
    add_topcoded_visit_group,
    compare_univariate_shape_models,
    cross_validated_skew_normal_vs_gmm,
    fixed_effect_group_comparisons,
    fixed_effect_logistic_adjusted_probabilities,
    fixed_effect_logistic_group_comparisons,
    gaussian_mixture_component_summary,
    gaussian_mixture_model_selection,
    gaussian_mixture_responsibilities,
    group_outcome_summary,
    parametric_bootstrap_skew_normal_vs_gmm,
    soft_component_composition,
    visit_frequency_cut_frontier,
    visit_frequency_support_audit,
)

GROUP_ORDER = ["0", "1", "2", "3", "4", "5", "6", "7+"]
USER_ORDER = ["1", "2", "3", "4", "5", "6", "7+"]
STATE_ORDER = ["pass", "numeric_nonpass", "BV", "RT", "BA"]

group_spec = json.loads((PAPER_DIR / "visit_grouping_spec.json").read_text(encoding="utf-8"))
group_spec

# Import the canonical publication-figure module so the notebook renders
# the same Matplotlib recipes as the manuscript instead of embedding PDFs.
CODE_DIR = REPO_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import figures_paper21 as paper21_figures
paper21_figures.set_style()



### 1.3 Full statistical output helpers

#### What this block does

The compact paper estimators return pairwise contrasts, omnibus tests, and model metadata. For the statistical-review version of the notebook we also want the complete fitted regression output, including every attendance coefficient, instructor-period fixed effect, degree-programme indicator, cluster-robust standard error, test statistic, $p$-value, and confidence interval.

#### What is happening mathematically / technically?

The helpers below refit exactly the same statsmodels formula and covariance specification used by the paper functions. For OLS/LPM,

$$
Y=X\beta+\varepsilon,
$$

and for binary outcomes under the logit specification,

$$
\operatorname{logit}\{P(Y_i=1\mid X_i)\}=X_i'\beta.
$$

The notebook therefore exposes both levels of output: the raw regression parameterization and the publication-level pairwise/omnibus contrasts. The nuisance fixed effects are not hidden, because this layer is intended for statistical audit rather than only stakeholder presentation.


In [ ]:
def _prepare_group_model_data(
    frame,
    *,
    group_col,
    group_order,
    outcome_col,
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=("CLAVECARRERA",),
):
    required = [
        group_col, outcome_col, fixed_effect_col, cluster_col,
        *categorical_covariates,
    ]
    d = frame.dropna(subset=required).copy()
    order = [str(x) for x in group_order]
    d[group_col] = d[group_col].astype(str)
    d = d.loc[d[group_col].isin(order)].copy()
    d[group_col] = pd.Categorical(d[group_col], categories=order, ordered=True)

    reference = order[0]
    group_term = f"C({group_col}, Treatment(reference='{reference}'))"
    formula = f"{outcome_col} ~ {group_term} + C({fixed_effect_col})"
    if categorical_covariates:
        formula += " + " + " + ".join(f"C({x})" for x in categorical_covariates)
    return d, formula


def fit_full_ols_group_model(
    frame,
    *,
    group_col,
    group_order,
    outcome_col,
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=("CLAVECARRERA",),
):
    d, formula = _prepare_group_model_data(
        frame,
        group_col=group_col,
        group_order=group_order,
        outcome_col=outcome_col,
        fixed_effect_col=fixed_effect_col,
        cluster_col=cluster_col,
        categorical_covariates=categorical_covariates,
    )
    model = smf.ols(formula, data=d).fit(
        cov_type="cluster",
        cov_kwds={"groups": d[cluster_col]},
    )
    return model, d, formula


def fit_full_logit_group_model(
    frame,
    *,
    group_col,
    group_order,
    outcome_col,
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=("CLAVECARRERA",),
):
    d, formula = _prepare_group_model_data(
        frame,
        group_col=group_col,
        group_order=group_order,
        outcome_col=outcome_col,
        fixed_effect_col=fixed_effect_col,
        cluster_col=cluster_col,
        categorical_covariates=categorical_covariates,
    )
    y = pd.to_numeric(d[outcome_col], errors="coerce")
    d = d.loc[y.isin([0, 1])].copy()
    d[outcome_col] = y.loc[d.index].astype(float)
    model = smf.glm(
        formula,
        data=d,
        family=sm.families.Binomial(),
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": d[cluster_col]},
        maxiter=200,
    )
    return model, d, formula


def full_coefficient_table(model, *, exponentiate=False):
    ci = np.asarray(model.conf_int(), dtype=float)
    out = pd.DataFrame({
        "term": model.params.index,
        "estimate": np.asarray(model.params, dtype=float),
        "cluster_robust_se": np.asarray(model.bse, dtype=float),
        "statistic": np.asarray(model.tvalues, dtype=float),
        "p_value": np.asarray(model.pvalues, dtype=float),
        "ci95_low": ci[:, 0],
        "ci95_high": ci[:, 1],
    })
    if exponentiate:
        out["odds_ratio"] = np.exp(out["estimate"])
        out["or_ci95_low"] = np.exp(out["ci95_low"])
        out["or_ci95_high"] = np.exp(out["ci95_high"])
    return out


def display_full_model(
    model,
    d,
    formula,
    *,
    cluster_col,
    title,
    exponentiate=False,
):
    metadata = {
        "formula": formula,
        "N": int(model.nobs),
        "df_model": float(model.df_model),
        "df_resid": float(model.df_resid),
        "covariance_type": model.cov_type,
        "cluster_variable": cluster_col,
        "n_clusters": int(d[cluster_col].nunique()),
        "log_likelihood": float(model.llf),
        "AIC": float(model.aic),
    }
    if hasattr(model, "rsquared"):
        metadata["R_squared"] = float(model.rsquared)
        metadata["adjusted_R_squared"] = float(model.rsquared_adj)
    if hasattr(model, "deviance"):
        metadata["deviance"] = float(model.deviance)
        metadata["null_deviance"] = float(model.null_deviance)
        metadata["pearson_chi2"] = float(model.pearson_chi2)
        metadata["converged"] = bool(model.converged)

    display(Markdown(f"#### {title} — model metadata"))
    display(pd.DataFrame([metadata]))
    display(Markdown(f"#### {title} — complete coefficient table"))
    display(full_coefficient_table(model, exponentiate=exponentiate))
    display(Markdown(f"#### {title} — native statsmodels summary"))
    display(model.summary2())


### Inline figure generation

Every manuscript figure below is generated by Matplotlib during the notebook run. The notebook imports the canonical functions from `code/figures_paper21.py`, but temporarily replaces their file-saving hook so that the live `Figure` object is rendered inline instead of loading a previously generated PDF.

This means the visual recipe is shared with the paper while the notebook itself performs the plotting.


### 1.3 Inline rendering of the canonical figure functions

#### What this block does

The paper plotting module normally saves each Matplotlib figure to disk and closes it. The notebook replaces only that final output hook so that the figure created by the canonical plotting function is displayed inline.

#### What is happening mathematically / technically?

This is a temporary **monkey patch** of `paper21_figures._save`. The plotting function still executes all of its original data reads, transformations, scales, annotations, and Matplotlib calls; only the terminal operation changes from

$$
\texttt{Figure}\rightarrow \texttt{savefig(PDF)}\rightarrow \texttt{close}
$$

to

$$
\texttt{Figure}\rightarrow \texttt{display in Jupyter}\rightarrow \texttt{close}.
$$

The original `_save` function is restored in a `finally` block, so a plotting exception cannot leave the imported module in a modified state. This keeps the notebook figures definitionally aligned with the publication figures.


In [ ]:
def render_paper_figure(plot_function):
    """Render one canonical Paper 2.1 Matplotlib figure inline.

    The publication plotting functions normally save a PDF and close the figure.
    Here we temporarily replace only that final output step; all data loading,
    transformations, annotations, scales, and plotting logic remain canonical.
    """
    original_save = paper21_figures._save

    def _capture(fig, name):
        fig.tight_layout()
        return fig

    paper21_figures._save = _capture
    try:
        fig = plot_function()
        display(fig)
        plt.close(fig)
        return None
    finally:
        paper21_figures._save = original_save


## 2. Institutional context and scientific question

CMAT is the mathematics support centre at Universidad de las Américas Puebla. Students register attendance when entering the centre, and each registration is counted as one visit. The records identify that a visit occurred, but they do not measure its duration, exact content, or the tutor involved.

The key institutional complication is PPA. During at least part of the study period, three CMAT visits could satisfy a first-semester PPA participation activity. Consequently, one, two, or three visits may combine academic demand with an institutional participation incentive. A fourth or later visit necessarily exceeds that specific three-visit option, although later attendance remains self-selected.

This motivates three interpretive regions:

$$
0 \quad | \quad 1,2,3 \quad | \quad 4+
$$

where the first region is no recorded CMAT use, the middle region is PPA-compatible low-frequency use, and the last region represents repeated attendance beyond that specific participation threshold. The manuscript also resolves a `7+` high-frequency tail when describing the full attendance profile.

The four substantive questions are:

1. Where do adjusted performance and PASS outcomes differ across attendance frequencies?
2. Is there evidence of a break exactly at the third visit?
3. Does the distribution of outcomes contain more than one performance regime?
4. Does the composition of adverse outcomes—numeric failure, BV, RT, and BA—differ across attendance patterns?


## 3. Data, cohort, and outcomes

The unit of analysis is the student's **first eligible observed attempt at Matemáticas Universitarias (MU)** in an academic period with CMAT attendance coverage. Visits are linked by student and academic period, regardless of the subject label recorded for the CMAT visit.

Two principal performance outcomes are used:

- **PASS**: numeric final grade $\ge 7.5$; numeric grade below 7.5, BV, RT, and BA are non-PASS.
- **Continuous standardized performance**: the completed final-grade value is standardized within instructor-period groups,

$$
Z_{ic}=\frac{Y_{ic}-\bar Y_c}{s_c},
$$

where $c$ denotes the instructor × academic-period grading context.

For the continuous primary outcome, administrative endings are numerically represented using the pre-specified Paper 2 imputation procedure. A numeric complete-case standardized outcome is retained as a sensitivity analysis so that the role of administrative-outcome imputation can be inspected explicitly.


### 3.1 Construction of the analytical cohort and outcomes

#### What this block does

This block reads the controlled pseudonymized course and CMAT-attendance files, applies the canonical study-cohort rules, constructs PASS and the continuous grade outcomes, preserves the exact administrative outcome type, and creates the main `0/1/2/3/4/5/6/7+` reporting variable.

#### What is happening mathematically / technically?

The cohort returned as `mu_primary` is the first eligible MU attempt per student under the shared study configuration. The outcome construction then proceeds in three distinct steps.

**1. Binary PASS.** For numeric grades,

$$
PASS_i=\mathbf 1(Y_i\ge 7.5),
$$

while records classified as adverse administrative endings are assigned $PASS_i=0$. Importantly, PASS itself does not depend on the imputed continuous value.

**2. Completion of the continuous grade.** For each `CLASSROOM_ID` (the instructor × academic-period grading context), the code collects the observed numeric grades strictly below 7.5. If that pool is non-degenerate, it fits a one-dimensional Gaussian KDE using SciPy's default Scott bandwidth factor and draws replacement values for BV/RT/BA by rejection sampling on $[0,7.5)$. If the pool is degenerate, it resamples empirically; if the classroom has no observed numeric sub-pass grade, it uses $U(0,7.5)$. Numeric grades are never altered.

Thus, for an adverse observation, the imputed grade is a draw from the **within-classroom empirical lower-tail model**, not a fixed penalty value.

**3. Within-classroom standardization.** After completion,

$$
Z_{ic}
=
\frac{Y^{*}_{ic}-\bar Y^{*}_{c}}
{s^{*}_{c}},
$$

where $c$ indexes `CLASSROOM_ID`, $\bar Y^{*}_{c}$ is the completed classroom mean, and $s^{*}_{c}$ is the sample standard deviation with $ddof=1$. A separate `Z_GRADE_COMPLETE_CASE` is computed analogously using only observed numeric grades.

Finally, `add_academic_outcome_states()` constructs exact states $\{PASS,\text{numeric non-pass},BV,RT,BA\}$ and binary contrasts that are defined only on the relevant subsets. The top-coding call maps $K_i>6$ to $7+$, while preserving exact values 0–6.


In [ ]:
# Load the exact controlled pseudonymized inputs used by the canonical Paper 2.1 runners.
base_config = get_study_config(REPO_ROOT)
config = replace(
    base_config,
    materias_path=(REPO_ROOT / "data" / "controlled" / "Materias_pseudonymized.csv").resolve(),
    asesorias_path=(REPO_ROOT / "data" / "controlled" / "Asesorias_pseudonymized.csv").resolve(),
)

data = load_and_clean_inputs(config)

mu = build_study_cohorts(data, config)["mu_primary"].copy()
mu = add_primary_outcomes(mu, config)
mu["PASS"] = mu["PASS"].astype(float)
mu = add_academic_outcome_states(mu)

# Main manuscript reporting grid: 0 / exact 1-6 / 7+.
mu = add_topcoded_visit_group(
    mu,
    visits_col="VISITS_CMAT_PERIOD",
    top_exact=6,
    include_zero=True,
    output_col="P21_GROUP_7P",
)

print(f"Study cohort N = {len(mu):,}")
print(f"CMAT users N = {int(mu['VISITS_CMAT_PERIOD'].gt(0).sum()):,}")
print(f"Instructor-period groups = {mu['CLASSROOM_ID'].nunique():,}")


## 4. Why the upper tail is grouped

The historical outcome-blind support audit selected `1 / 2 / 3 / 4 / 5 / 6+` as the most conservative positive-attendance inferential grouping. The current manuscript additionally displays exact 6 and 7+ because the scientific question concerns high-frequency use, but this extra resolution must not be rewritten as if it had been pre-specified from outcome significance.

The governing principle is support-preserving resolution: retain exact visit counts while student counts and overlap across instructor-period grading contexts remain adequate; pool once exact-frequency cells become too sparse for stable adjusted comparisons.


### 4.1 Outcome-blind support and overlap audit

#### What this block does

This block displays the tables used to decide how finely visit frequency can be resolved before inspecting academic outcomes. The decision is based on cell size and within-context overlap, not on which grouping gives the smallest $p$-value.

#### What is happening mathematically / technically?

For each exact positive visit count $k$, the support audit records

$$
n_k,\qquad
C_k=\#\{c:\exists i\text{ in context }c\text{ with }K_i=k\},
$$

plus the number of instructors and the size of the upper tail.

The overlap diagnostic treats the set of instructor-period contexts containing group $k$ as

$$
\mathcal C_k=\{c: K_i=k\text{ for at least one }i\in c\}.
$$

For every pair $(k,j)$, the relevant overlap is

$$
O_{kj}=|\mathcal C_k\cap\mathcal C_j|.
$$

This matters because a fixed-effect comparison is most informative when different exposure groups coexist within the same fixed-effect strata. A frequency cell can have a non-trivial total $n$ but still provide weak conditional support if it appears in contexts where comparison groups are absent.

For each candidate top-exact cutoff $q$, the frontier therefore evaluates the minimum and median overlap among adjacent and all pairs after defining $1,\ldots,q,(q+1)+$. The historical `6+` choice came from this outcome-blind support exercise; exact 6 / 7+ is a higher-resolution reporting sensitivity.


In [ ]:
support = visit_frequency_support_audit(
    mu,
    visits_col="VISITS_CMAT_PERIOD",
    cluster_col="CLASSROOM_ID",
    instructor_col="CLAVEPROFESOR",
)

overlap_frontier = visit_frequency_cut_frontier(
    mu,
    visits_col="VISITS_CMAT_PERIOD",
    cluster_col="CLASSROOM_ID",
    min_top_exact=1,
    max_top_exact=10,
)

display(Markdown("#### Exact positive-frequency support — recalculated in notebook"))
display(support)

display(Markdown("#### Candidate exact-plus-tail overlap frontier — recalculated in notebook"))
display(overlap_frontier)



The important feature is not only the number of students in a visit-frequency cell, but also whether adjacent frequency groups coexist across enough instructor-period contexts to support fixed-effect comparisons. Exact six is already relatively sparse; exact seven, eight, and nine are substantially smaller. Thus `7+` should be interpreted as a high-frequency descriptive tail, not as a uniquely optimal scientific threshold.


## 5. Descriptive attendance profile

The full manuscript reporting grid contains 5,393 students with zero visits and 1,234 students with at least one visit. Positive attendance is concentrated at low frequencies, with a smaller high-use tail.


### 5.1 Unadjusted descriptive summaries by attendance frequency

#### What this block does

This block computes the raw group means and PASS rates for the eight reporting groups before fitting any regression model.

#### What is happening mathematically / technically?

For each attendance group $k$, `group_outcome_summary()` computes

$$
\bar Z_k=\frac{1}{n_k}\sum_{i:K_i=k}Z_i,
\qquad
s_k^2=\frac{1}{n_k-1}\sum_{i:K_i=k}(Z_i-\bar Z_k)^2,
$$

and reports the conventional normal-approximation interval

$$
\bar Z_k\pm 1.96\frac{s_k}{\sqrt{n_k}}.
$$

For PASS it computes

$$
\hat p_k=\frac{1}{n_k}\sum_{i:K_i=k}PASS_i,
\qquad
SE(\hat p_k)=
\sqrt{\frac{\hat p_k(1-\hat p_k)}{n_k}},
$$

with the corresponding Wald interval clipped to $[0,1]$.

These intervals are **not cluster-robust** and do not adjust for `CLASSROOM_ID` or degree programme. They are descriptive summaries only; the conditional comparisons begin in the regression blocks below.


In [ ]:
descriptives = group_outcome_summary(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="Z_GRADE_PRIMARY",
    pass_col="PASS",
)

descriptives = descriptives.rename(columns={
    "outcome_mean": "mean_z",
    "outcome_sd": "sd_z",
    "outcome_ci95_low": "z_ci95_low",
    "outcome_ci95_high": "z_ci95_high",
    "binary_rate": "pass_rate",
    "binary_ci95_low": "pass_ci95_low",
    "binary_ci95_high": "pass_ci95_high",
})

display(
    descriptives[
        ["group", "n", "mean_z", "z_ci95_low", "z_ci95_high",
         "pass_rate", "pass_ci95_low", "pass_ci95_high"]
    ]
)


The descriptive profile already shows the main empirical pattern. Mean standardized performance is approximately $-0.061$ at zero attendance and is positive in every attendance group. The corresponding observed pass rates are about 72.1% at zero visits, around 81% for one to three visits, and 89.1% for the 7+ group.

The sequence is **not monotone**. For example, the exact-six point estimate is below the five-visit estimate and below the 7+ estimate. This is one reason not to read visit count as a simple treatment dose.


### 5.2 Generation of the descriptive distribution figures

#### What this block does

The first figure displays the observed numeric-grade distribution together with the administrative outcome shares before any continuous-outcome imputation. The second displays the completed standardized outcome and decomposes its density by final academic-result state.

#### What is happening mathematically / technically?

The first panel family uses 0.1-grade bins. Each bin count is divided by the **full attendance-group size**, not only by the number of numeric-grade observations. Therefore, the integral/mass of the numeric bars corresponds to the numeric share of that attendance group, while BV/RT/BA account for the remaining probability mass.

The ridgeline figure uses the precomputed `mixture_component_density()` output. Within each attendance group, all outcome-state KDE components use a common bandwidth and are weighted by their share of the full group, so

$$
\hat f_k(z)
=
\sum_{s\in\mathcal S}
\hat p_{ks}\,\hat f_{ks}(z),
$$

where $s\in\{PASS,\text{numeric fail},BV,RT,BA\}$. Hence the stacked components sum to the total group density and the area of each component reflects its observed group share.

This visualization is descriptive; it is not the Gaussian-mixture model introduced later.


In [ ]:
# Figure 1a: observed numeric grades and administrative outcomes before imputation.
render_paper_figure(paper21_figures.plot_observed_pre_imputation_structure)

# Figure 1: completed standardized outcome and outcome-state composition.
render_paper_figure(paper21_figures.plot_distribution_and_composition)


## 6. Primary benchmark: zero visits versus any attendance

The first estimand deliberately collapses all positive attendance into one group. This isolates **entry into the support system** from the later question of visit frequency among users.

The benchmark estimated in Section 6.1 is an OLS regression that compares zero visits with any recorded attendance while including instructor-period fixed effects and degree-programme indicators; its covariance matrix is clustered by instructor-period. For the continuous outcome, the attendance contrast is expressed in within-instructor-period standard-deviation units. For PASS, the same regression is used as a linear-probability model, so the attendance contrast is an adjusted probability difference that can be reported in percentage points.


### 6.1 Benchmark model: zero visits versus any recorded attendance

#### What this block does

This block creates a binary exposure $\mathbb 1(K_i>0)$ and fits the paper's primary benchmark twice: once with `Z_GRADE_PRIMARY` as the outcome and once with PASS as the outcome.

#### What is happening mathematically / technically?

The call to `fixed_effect_group_comparisons()` constructs the exact statsmodels formula

$$
Y_i
=
\beta_0
+\beta_A A_i
+\sum_{c=2}^{C}\alpha_c\,\mathbf 1(C_i=c)
+\sum_{d=2}^{D}\delta_d\,\mathbf 1(D_i=d)
+\varepsilon_i,
$$

where

- $A_i=\mathbf 1(K_i>0)$;
- $C_i$ is `CLASSROOM_ID` = instructor × academic period;
- $D_i$ is `CLAVECARRERA`;
- $Y_i$ is either $Z_i$ or $PASS_i$.

This is implemented as **ordinary least squares with explicit dummy variables**, not as a mixed model and not as a conditional fixed-effects estimator. For PASS, OLS on a 0/1 outcome is the linear probability model (LPM).

The point estimate is the OLS projection

$$
\hat\beta=(X'X)^{-1}X'Y
$$

(up to the generalized-inverse handling used internally for any redundant dummy columns). Uncertainty is replaced by a cluster-robust sandwich covariance at `CLASSROOM_ID`,

$$
\widehat V_{\mathrm{CR}}
\propto
(X'X)^{-1}
\left(
\sum_c X_c'\hat u_c\hat u_c'X_c
\right)
(X'X)^{-1},
$$

with statsmodels' cluster-covariance implementation supplying the finite-sample conventions.

Because the group order is $[0,1+]$, the underlying regression coefficient is parameterized as $1+$ relative to 0, but `fixed_effect_group_comparisons()` reports the requested pair as $\widehat{0-(1+)}$. The wrapper deliberately negates that quantity so the displayed estimand is

$$
\widehat\Delta_{1+,0}
=
\widehat E[Y\mid 1+,X]
-
\widehat E[Y\mid 0,X].
$$

For the standardized continuous outcome, this is measured in within-classroom SD units. For the LPM, it is a probability difference, so multiplying by 100 gives percentage points. The fixed effects and degree indicators adjust observed composition; they do not remove unmeasured selection into CMAT.


In [ ]:
benchmark_data = mu.copy()
benchmark_data["ATTENDANCE_BINARY"] = np.where(
    benchmark_data["VISITS_CMAT_PERIOD"].gt(0), "1+", "0"
)

def run_binary_benchmark(outcome_col, title):
    pairwise, omnibus, info = fixed_effect_group_comparisons(
        benchmark_data,
        group_col="ATTENDANCE_BINARY",
        group_order=["0", "1+"],
        outcome_col=outcome_col,
        fixed_effect_col="CLASSROOM_ID",
        cluster_col="CLASSROOM_ID",
        categorical_covariates=["CLAVECARRERA"],
        multiplicity_method="holm",
    )

    model, d, formula = fit_full_ols_group_model(
        benchmark_data,
        group_col="ATTENDANCE_BINARY",
        group_order=["0", "1+"],
        outcome_col=outcome_col,
        cluster_col="CLASSROOM_ID",
    )
    display_full_model(
        model, d, formula,
        cluster_col="CLASSROOM_ID",
        title=title,
    )

    display(Markdown(f"#### {title} — pairwise contrast"))
    display(pairwise)
    display(Markdown(f"#### {title} — omnibus test"))
    display(omnibus)
    display(Markdown(f"#### {title} — compact model information"))
    display(info)

    row = pairwise.iloc[0]
    return {
        "outcome": outcome_col,
        "1+ minus 0": -float(row["estimate_group1_minus_group2"]),
        "cluster_robust_se": float(row["cluster_robust_se"]),
        "ci95_low": -float(row["ci_high"]),
        "ci95_high": -float(row["ci_low"]),
        "p_value": float(row["p_raw"]),
        "N": int(row["n"]),
        "clusters": int(info.loc[0, "n_clusters"]),
    }

benchmark = pd.DataFrame([
    run_binary_benchmark(
        "Z_GRADE_PRIMARY",
        "Continuous standardized grade: any attendance vs zero",
    ),
    run_binary_benchmark(
        "PASS",
        "PASS linear-probability model: any attendance vs zero",
    ),
])

display(Markdown("#### Compact benchmark table used in the manuscript"))
display(benchmark)



The central benchmark is large: any recorded attendance is associated with approximately **+0.361 SD** in standardized final performance and **+15.3 percentage points** in pass probability. These are adjusted associations, not causal treatment effects.

Formally, the fitted comparison can be written schematically as

$$
Y_i=\alpha+\beta\,\mathbb{1}(\mathrm{CMAT}_i>0)
+\gamma_{\mathrm{instructor}\times\mathrm{period}}
+\delta_{\mathrm{degree}}+\varepsilon_i.
$$

The coefficient $\beta$ describes an adjusted difference in observed outcomes. Because CMAT use is selected rather than randomized, it is not automatically equal to a causal potential-outcome contrast.


## 7. Frequency among users and the PPA threshold

The next question is whether outcomes form a staircase across 0, 1, 2, ..., 7+ visits. All 28 pairwise comparisons in the eight-group family are estimated with instructor-period fixed effects, degree-programme adjustment, cluster-robust uncertainty, and Holm correction for multiple testing.

The third visit receives special attention because it is the institutional PPA threshold, but an institutional threshold need not be an academic performance threshold.


### 7.1 Eight-group fixed-effect model and the full pairwise contrast family

#### What this block does

This block replaces the binary attendance indicator with the complete reporting factor `0/1/2/3/4/5/6/7+` and fits the same adjusted model separately for standardized performance and PASS.

#### What is happening mathematically / technically?

With 0 as the treatment reference, the fitted OLS model is

$$
Y_i
=
\beta_0
+\sum_{k\in\{1,2,3,4,5,6,7+\}}
\beta_k\,\mathbf 1(K_i=k)
+\sum_{c=2}^{C}\alpha_c\mathbf 1(C_i=c)
+\sum_{d=2}^{D}\delta_d\mathbf 1(D_i=d)
+\varepsilon_i.
$$

Again, PASS is fit as an LPM; the continuous outcome uses the same design matrix. Cluster-robust covariance is computed by `CLASSROOM_ID`.

For any two groups $k$ and $j$, the code creates a contrast vector $a_{kj}$ and asks statsmodels for

$$
\widehat\Delta_{kj}
=
a_{kj}'\hat\beta,
\qquad
SE(\widehat\Delta_{kj})
=
\sqrt{a_{kj}'\widehat V_{\mathrm{CR}}a_{kj}}.
$$

If neither group is the reference, this is algebraically $\hat\beta_k-\hat\beta_j$. With eight groups there are

$$
\binom{8}{2}=28
$$

pairwise tests.

The omnibus test is the joint null

$$
H_0:
\beta_1=\beta_2=\cdots=\beta_{7+}=0,
$$

implemented by `model.f_test(R\beta=0)` using the fitted covariance structure.

Finally, the 28 raw pairwise $p$-values are adjusted using Holm's step-down family-wise-error procedure. If $p_{(1)}\le\cdots\le p_{(m)}$, Holm sequentially compares $p_{(r)}$ with $\alpha/(m-r+1)$. This is why a locally small raw $p$-value need not remain significant after the full comparison family is acknowledged.


In [ ]:
z_pair, z_omnibus, z_info = fixed_effect_group_comparisons(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="Z_GRADE_PRIMARY",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
    multiplicity_method="holm",
)

pass_pair, pass_omnibus, pass_info = fixed_effect_group_comparisons(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="PASS",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
    multiplicity_method="holm",
)

z_model, z_model_data, z_formula = fit_full_ols_group_model(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="Z_GRADE_PRIMARY",
    cluster_col="CLASSROOM_ID",
)
pass_lpm_model, pass_lpm_data, pass_lpm_formula = fit_full_ols_group_model(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="PASS",
    cluster_col="CLASSROOM_ID",
)

display_full_model(
    z_model, z_model_data, z_formula,
    cluster_col="CLASSROOM_ID",
    title="Eight-group continuous-outcome OLS",
)
display(Markdown("#### Continuous outcome — all 28 pairwise contrasts"))
display(z_pair)
display(Markdown("#### Continuous outcome — omnibus test"))
display(z_omnibus)
display(Markdown("#### Continuous outcome — compact model information"))
display(z_info)

display_full_model(
    pass_lpm_model, pass_lpm_data, pass_lpm_formula,
    cluster_col="CLASSROOM_ID",
    title="Eight-group PASS linear-probability model",
)
display(Markdown("#### PASS LPM — all 28 pairwise contrasts"))
display(pass_pair)
display(Markdown("#### PASS LPM — omnibus test"))
display(pass_omnibus)
display(Markdown("#### PASS LPM — compact model information"))
display(pass_info)



### 7.2 Extraction of pre-interpreted contrasts from the fitted family

#### What this block does

This helper retrieves specific rows from the already fitted 28-comparison table: 2 vs 3, 4 vs 5, and 6 vs 7+.

#### What is happening mathematically / technically?

No model is estimated here. The function only searches the long-format contrast table while allowing either orientation $(k,j)$ or $(j,k)$.

This distinction matters because the scientific interpretation is attached to particular linear hypotheses:

$$
H_0^{PPA}:\mu_2-\mu_3=0,
$$

$$
H_0^{4\rightarrow5}:\mu_4-\mu_5=0,
\qquad
H_0^{6\rightarrow7+}:\mu_6-\mu_{7+}=0.
$$

The estimates, cluster-robust standard errors, confidence intervals, raw $p$-values, and Holm-adjusted $p$-values all come from the single model fitted in the previous block. This cell merely exposes the contrasts that are substantively relevant to the PPA and high-frequency hypotheses.


In [ ]:
def get_pair(frame, a, b):
    hit = frame.loc[
        ((frame["group1"].astype(str) == str(a)) & (frame["group2"].astype(str) == str(b))) |
        ((frame["group1"].astype(str) == str(b)) & (frame["group2"].astype(str) == str(a)))
    ].copy()
    if hit.empty:
        raise KeyError(f"Pair {a} vs {b} not found")
    return hit

display(Markdown("### PPA threshold: 2 versus 3 visits"))
display(get_pair(z_pair, "2", "3"))

display(Markdown("### Descriptive later changes: 4 versus 5 and 6 versus 7+"))
display(pd.concat([
    get_pair(z_pair, "4", "5"),
    get_pair(z_pair, "6", "7+"),
], ignore_index=True))


The two-versus-three contrast is extremely small and statistically unresolved, so the third visit is not an observable performance discontinuity. The visually noticeable increases from four to five visits and from six to 7+ are also not statistically resolved after accounting for uncertainty and multiplicity.

This produces a central distinction:

$$
\text{institutional threshold at 3 visits}
\neq
\text{demonstrated academic threshold at 3 visits}.
$$

Likewise, visible changes later in the sequence should be described as **descriptive profile changes**, not causal jumps.


## 8. Why 7+ is nevertheless distinctive

The 7+ group has the highest descriptive mean standardized outcome and the highest observed pass rate. The binary PASS analysis provides stronger within-user evidence than the continuous outcome: the adjusted pass probability for 7+ exceeds the one-visit group by about 14.5 percentage points after Holm correction.

The manuscript also reports fixed-effect logistic **odds-ratio point estimates** for magnitude. Because the full logistic clustered covariance is rank-deficient in this design, multiplicity-aware PASS inference comes from the stable linear-probability model, while logistic odds ratios are treated as descriptive effect-size estimates.


### 8.1 High-frequency PASS: LPM inference and fixed-effect logit odds ratios

#### What this block does

This block first displays the 1 vs 7+ PASS contrast from the LPM fitted above, then fits a separate binomial-logit model to express pairwise differences on the odds-ratio scale.

#### What is happening mathematically / technically?

The logistic specification is

$$
\log\frac{P(PASS_i=1\mid X_i)}
{1-P(PASS_i=1\mid X_i)}
=
\beta_0
+\sum_k\beta_k\mathbf 1(K_i=k)
+\sum_c\alpha_c\mathbf 1(C_i=c)
+\sum_d\delta_d\mathbf 1(D_i=d).
$$

In code this is `statsmodels.glm(..., family=Binomial())` with the same explicit fixed-effect dummies and cluster-robust covariance by `CLASSROOM_ID`. The GLM coefficients are estimated by the numerical fitting routine used by statsmodels for the binomial likelihood; they are not conditional-logit fixed effects.

For a pair $k,j$,

$$
\widehat{\log OR}_{kj}
=
a_{kj}'\hat\beta,
\qquad
\widehat{OR}_{kj}
=
\exp(a_{kj}'\hat\beta).
$$

Thus the reported OR is a **conditional model odds ratio**, not a risk ratio and not the same estimand as the LPM percentage-point contrast.

A technical problem arises in the omnibus clustered logit covariance: with many fixed-effect dummies and sparse high-frequency cells, the restricted covariance can be rank-deficient. The library therefore computes the omnibus Wald statistic with a Moore-Penrose inverse,

$$
W=(R\hat\beta)'
(R\widehat V R')^{+}
(R\hat\beta),
$$

and uses $\mathrm{rank}(R\widehat V R')$ as the chi-square degrees of freedom. Because this covariance issue makes multiplicity-aware logistic inference less stable, Paper 2.1 uses the LPM pairwise tests for PASS significance and retains logit primarily for OR effect-size description.


In [ ]:
display(Markdown("#### 1 versus 7+ PASS contrast from the linear-probability model"))
display(get_pair(pass_pair, "1", "7+"))

logit_pair, logit_omnibus, logit_info = fixed_effect_logistic_group_comparisons(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="PASS",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
    multiplicity_method="holm",
)

pass_logit_model, pass_logit_data, pass_logit_formula = fit_full_logit_group_model(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="PASS",
    cluster_col="CLASSROOM_ID",
)

display_full_model(
    pass_logit_model, pass_logit_data, pass_logit_formula,
    cluster_col="CLASSROOM_ID",
    title="Eight-group PASS binomial-logit model",
    exponentiate=True,
)

display(Markdown("#### PASS logit — all pairwise odds-ratio contrasts"))
display(logit_pair)
display(Markdown("#### PASS logit — omnibus Wald test"))
display(logit_omnibus)
display(Markdown("#### PASS logit — compact model information"))
display(logit_info)
display(Markdown("#### PASS logit — 1 versus 7+ contrast"))
display(get_pair(logit_pair, "1", "7+"))



### 8.2 Pairwise effect and multiplicity dashboards

#### What this block does

The plotting functions reorganize the long-format pairwise output into symmetric $8\times8$ matrices and render effect sizes, raw $p$-values, and Holm-adjusted $p$-values.

#### What is happening mathematically / technically?

For the continuous outcome, matrix entry $(k,j)$ contains

$$
M^{Z}_{kj}=\widehat\Delta_{kj},
\qquad
M^{Z}_{jk}=-M^{Z}_{kj}.
$$

For PASS odds ratios,

$$
M^{OR}_{kj}=\widehat{OR}_{kj},
\qquad
M^{OR}_{jk}=1/\widehat{OR}_{kj}.
$$

The plotting code log-transforms the OR matrix for a color scale centered symmetrically at the null because

$$
\log(OR_{jk})=-\log(OR_{kj}),
$$

while it prints the OR itself as the cell annotation.

The $p$-value matrices are symmetric because the two-sided test of $k-j=0$ is identical to that of $j-k=0$. Raw and Holm-adjusted values are displayed separately so magnitude, local evidence, and family-wise inference are not conflated.


In [ ]:
# Pairwise adjusted effect dashboard.
render_paper_figure(paper21_figures.plot_pairwise_effect_dashboard)

# Raw and Holm-adjusted pairwise p-value dashboards.
render_paper_figure(paper21_figures.plot_z_pvalue_dashboard)
render_paper_figure(paper21_figures.plot_pass_pvalue_dashboard)


The evidence therefore supports describing 7+ as a **distinctive high-frequency profile**, especially for PASS. It does not support the stronger claim that a causal threshold begins at the seventh visit, because the adjacent exact-six versus 7+ comparison is itself imprecise.


## 9. Adverse outcomes are not interchangeable

PASS/non-PASS hides substantively different endpoints. A non-passing attempt may end as:

- a completed numeric grade below 7.5,
- BV (voluntary withdrawal),
- RT (temporary withdrawal),
- BA (academic withdrawal).

The paper therefore distinguishes a **performance margin** from an **academic-management margin**. The second asks, conditional on not passing, how the adverse outcome is recorded.


### 9.1 Five-state decomposition of the final academic record

#### What this block does

This block replaces the binary PASS/non-PASS summary with the exact five-state outcome: PASS, numeric grade below 7.5, BV, RT, and BA.

#### What is happening mathematically / technically?

For each attendance group $k$ and outcome state $s$, the code estimates the empirical multinomial cell probability

$$
\hat p_{ks}
=
\frac{n_{ks}}{n_k},
\qquad
\sum_s\hat p_{ks}=1.
$$

No multinomial regression is fitted in this block. It is a direct contingency-table decomposition using the exact administrative token preserved by `add_academic_outcome_states()`.

The distinction is essential because two groups can have the same

$$
P(nonPASS\mid K=k)
$$

while having different conditional compositions

$$
P(S=s\mid nonPASS,K=k).
$$

The next block moves from this unconditional state composition to explicit binary contrasts within the non-PASS subset.


In [ ]:
composition = (
    mu.groupby(["P21_GROUP_7P", "ACADEMIC_OUTCOME_STATE_5"], observed=True)
      .size()
      .rename("n")
      .reset_index()
)
composition["share"] = composition.groupby("P21_GROUP_7P")["n"].transform(lambda s: s / s.sum())
composition_pivot = composition.pivot(
    index="P21_GROUP_7P", columns="ACADEMIC_OUTCOME_STATE_5", values="share"
).reindex(GROUP_ORDER).fillna(0)

composition_pivot


### 9.2 Conditional academic-management contrasts among non-PASS observations

#### What this block does

This block restricts the analysis to $PASS=0$ and estimates whether the **type** of adverse ending differs between zero attendance and any attendance.

#### What is happening mathematically / technically?

`add_academic_outcome_states()` creates each contrast with deliberate missingness. For example,

$$
BV\_VS\_NUMERIC_i=
\begin{cases}
1,&S_i=BV,\\
0,&S_i=\text{numeric non-pass},\\
NA,&S_i\in\{RT,BA\}.
\end{cases}
$$

Therefore, when `fixed_effect_group_comparisons()` drops missing model rows, the BV model is fitted **only** to BV and numeric-failure observations. The RT comparison analogously excludes BV and BA. The broad administrative contrast codes BV/RT/BA as 1 and numeric failure as 0.

For each binary contrast $M_i$, the fitted model is an LPM,

$$
M_i
=
\beta_0+\beta_A A_i
+\alpha_{C_i}
+\delta_{D_i}
+\varepsilon_i,
$$

on the relevant non-PASS comparison set, with cluster-robust covariance by `CLASSROOM_ID`.

Thus $\beta_A$ estimates an adjusted **difference in conditional composition**, e.g.

$$
P(BV\mid BV\text{ or numeric fail},A=1,X)
-
P(BV\mid BV\text{ or numeric fail},A=0,X),
$$

within the linear projection. It is not the probability of BV in the full cohort, and conditioning on non-PASS means it should not be interpreted as a total causal effect of CMAT.


In [ ]:
nonpass = mu.loc[mu["PASS"].eq(0)].copy()

def management_benchmark(outcome_col: str):
    x = nonpass.loc[nonpass[outcome_col].notna()].copy()
    x["ATTENDANCE_BINARY"] = np.where(x["VISITS_CMAT_PERIOD"].gt(0), "1+", "0")
    pairwise, _, info = fixed_effect_group_comparisons(
        x,
        group_col="ATTENDANCE_BINARY",
        group_order=["0", "1+"],
        outcome_col=outcome_col,
        fixed_effect_col="CLASSROOM_ID",
        cluster_col="CLASSROOM_ID",
        categorical_covariates=["CLAVECARRERA"],
        multiplicity_method="holm",
    )
    row = pairwise.iloc[0]
    return pd.Series({
        "contrast": outcome_col,
        "1+ minus 0": -float(row["estimate_group1_minus_group2"]),
        "ci95_low": -float(row["ci_high"]),
        "ci95_high": -float(row["ci_low"]),
        "p_value": float(row["p_raw"]),
        "N": int(row["n"]),
    })

management = pd.DataFrame([
    management_benchmark("ADMINISTRATIVE_VS_NUMERIC_NONPASS"),
    management_benchmark("BV_VS_NUMERIC_NONPASS"),
    management_benchmark("RT_VS_NUMERIC_NONPASS"),
])
management


Among students who do not pass, CMAT users are more likely to end with an administrative outcome rather than a numeric failure. Crucially, that pattern is concentrated in **BV**. The corresponding RT-versus-numeric-failure contrast is essentially null.

This prevents the interpretation from collapsing all administrative outcomes into one behavioral mechanism. The records show a BV-specific composition difference; they do **not** show why students used BV, whether they knew particular institutional rules, or whether CMAT changed the withdrawal decision.


## 10. Numeric failure without administrative-outcome imputation

A useful robustness check removes BV, RT, and BA entirely and asks a simpler question among students with an observed numeric final grade:

$$
P(Y<7.5\mid \text{visit frequency}).
$$

If the broad zero-versus-positive difference survives here, it cannot be attributed solely to how administrative outcomes were numerically imputed into the continuous outcome.


### 10.1 Numeric complete-case sensitivity

#### What this block does

This block removes all administrative outcomes and examines only observed numeric final grades. The binary variable `NUMERIC_FAIL` equals one when the observed numeric grade is below 7.5.

#### What is happening mathematically / technically?

The analysis sample becomes

$$
\mathcal I_{\mathrm{num}}
=
\{i:\texttt{GRADE\_CLASS}_i=\text{numeric}\}.
$$

Within that sample,

$$
NUMERIC\_FAIL_i
=
1-PASS_i
=
\mathbf 1(Y_i<7.5).
$$

The displayed table estimates the unadjusted empirical conditional risk

$$
\hat p_k^{num}
=
\frac{\sum_{i\in\mathcal I_{\mathrm{num}}}
\mathbf 1(K_i=k)\,NUMERIC\_FAIL_i}
{\sum_{i\in\mathcal I_{\mathrm{num}}}\mathbf 1(K_i=k)}.
$$

No imputed grade enters this outcome. This sensitivity therefore isolates whether the broad attendance pattern remains visible among students whose course record ended in an observed numeric grade.


In [ ]:
numeric = mu.loc[mu["GRADE_CLASS"].eq("numeric")].copy()
numeric["NUMERIC_FAIL"] = 1.0 - numeric["PASS"]

numeric_fail_summary = (
    numeric.groupby("P21_GROUP_7P", observed=True)["NUMERIC_FAIL"]
           .agg(n="size", failures="sum", probability="mean")
           .reindex(GROUP_ORDER)
)
numeric_fail_summary


### 10.2 Model-standardized numeric-failure probabilities from fixed-effect logit

#### What this block does

This block fits a binomial logit for numeric failure and converts the fitted model into adjusted probabilities for each attendance-frequency group.

#### What is happening mathematically / technically?

The fitted model is

$$
\operatorname{logit}
P(NUMERIC\_FAIL_i=1\mid K_i,C_i,D_i)
=
\beta_0+\beta_{K_i}+\alpha_{C_i}+\delta_{D_i}.
$$

The important detail is that `fixed_effect_logistic_adjusted_probabilities()` does **not** report the inverse-logit of a coefficient evaluated at one arbitrary reference profile. It performs empirical standardization.

For each group $k$, it copies the entire fitted-model sample and counterfactually replaces every observation's attendance group with $k$, while retaining that observation's actual classroom fixed-effect level and degree programme. It predicts

$$
\hat p_i(k)
=
\operatorname{logit}^{-1}(x_i(k)'\hat\beta)
$$

for every row and averages:

$$
\hat P_k^{std}
=
\frac{1}{n}\sum_{i=1}^{n}\hat p_i(k).
$$

Hence each frequency is standardized over the **same empirical distribution of classroom contexts and degree programmes**.

The standard error is obtained by the delta method. If

$$
g_k(\beta)=\frac1n\sum_i
\operatorname{logit}^{-1}(x_i(k)'\beta),
$$

the code computes

$$
\nabla g_k(\hat\beta)
=
\frac1n\sum_i
\hat p_i(k)\{1-\hat p_i(k)\}x_i(k)
$$

and then

$$
SE(\hat P_k^{std})
=
\sqrt{
\nabla g_k'
\widehat V_{\mathrm{cluster}}
\nabla g_k
}.
$$

This is substantially more informative than saying only that the probabilities are “adjusted.”


In [ ]:
adjusted_failure = fixed_effect_logistic_adjusted_probabilities(
    numeric,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="NUMERIC_FAIL",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
)
adjusted_failure


The observed-numeric analysis preserves the same broad structure: completed numeric failure is much more common at zero attendance than among CMAT users. Among positive frequencies, however, there is no regular visit-by-visit failure gradient. This is another reason the paper separates the large **engagement margin** (0 vs 1+) from the weaker **frequency margin** among users.


## 11. Distributional question: one asymmetric distribution or two regimes?

A mean can hide substantial heterogeneity. The paper therefore asks whether each attendance group is better represented by a single asymmetric distribution or by a two-component Gaussian mixture.

For a two-Gaussian model,

$$
f(z)=\pi\,\phi(z;\mu_L,\sigma_L^2)
+(1-\pi)\,\phi(z;\mu_H,\sigma_H^2),
$$

where the labels $L$ and $H$ refer only to the lower-mean and higher-mean density components. They are **not** assumed to be literal latent student types.

The critical alternative is a one-component skew-normal distribution. Without this comparison, ordinary left-skewness could be mistaken for substantive bimodality.


### Heavy-model execution policy

The complete mixture analysis includes multistart Gaussian mixtures, a skew-normal comparison, parametric bootstrap, and repeated cross-validation. Those calculations are intentionally expensive. The repository already contains the verified aggregate outputs used by the manuscript.

Set `RUN_HEAVY_MODELS = True` below only when a full recomputation is desired. Otherwise, the notebook reads the canonical versioned outputs.


### 11.1 Optional full re-estimation of the distributional models

#### What this block does

When enabled, this block reruns the expensive Paper 2.1 mixture pipeline: Gaussian mixtures with $K=1,2,3$, skew-normal alternatives, parametric bootstrap diagnostics, and repeated held-out predictive comparisons.

#### What is happening mathematically / technically?

For a $K$-component univariate Gaussian mixture,

$$
f(z_i\mid\theta)
=
\sum_{k=1}^{K}
\pi_k\,
\phi(z_i;\mu_k,\sigma_k^2),
\qquad
\pi_k\ge0,\quad \sum_k\pi_k=1.
$$

The log-likelihood is

$$
\ell(\theta)
=
\sum_{i=1}^{n}
\log\left[
\sum_{k=1}^{K}
\pi_k\phi(z_i;\mu_k,\sigma_k^2)
\right].
$$

`sklearn.mixture.GaussianMixture` maximizes this likelihood with **Expectation-Maximization (EM)**. At iteration $t$, the E-step computes posterior responsibilities

$$
r_{ik}^{(t)}
=
P(C_i=k\mid z_i,\theta^{(t)})
=
\frac{
\pi_k^{(t)}
\phi(z_i;\mu_k^{(t)},\sigma_k^{2(t)})
}{
\sum_h
\pi_h^{(t)}
\phi(z_i;\mu_h^{(t)},\sigma_h^{2(t)})
},
$$

and the M-step updates

$$
N_k^{(t)}=\sum_i r_{ik}^{(t)},
\qquad
\pi_k^{(t+1)}=\frac{N_k^{(t)}}{n},
$$

$$
\mu_k^{(t+1)}
=
\frac{\sum_i r_{ik}^{(t)}z_i}{N_k^{(t)}},
$$

$$
\sigma_k^{2(t+1)}
=
\frac{\sum_i r_{ik}^{(t)}
(z_i-\mu_k^{(t+1)})^2}{N_k^{(t)}}
+\lambda,
$$

where the implementation uses covariance regularization $\lambda=10^{-6}$. Candidate fits use `max_iter=1000` and `tol=1e-6`.

Because mixture likelihoods are non-convex, the runner uses multiple EM starts. For $K=2$, default randomized starts are supplemented by the scientifically proposed mean start $(-1.1,0.5)$ and an empirical interquartile start $(Q_{.25},Q_{.75})$; the converged candidate with the largest likelihood is retained.

Model selection is not based on likelihood alone. For each $K$,

$$
BIC=-2\ell(\hat\theta_K)+p_K\log n,
$$

and

$$
ICL=BIC+2H,
\qquad
H=-\sum_{i,k}r_{ik}\log r_{ik},
$$

so ICL additionally penalizes uncertain component classification.

The usual chi-square LRT reference for $K=1$ vs $K=2$ is invalid because mixture models violate regularity conditions at the boundary. The code therefore simulates under the fitted $K=1$ Gaussian null and obtains an empirical bootstrap $p$-value for

$$
LR=2\{\ell_{K=2}-\ell_{K=1}\}.
$$

The stronger shape check uses a **fitted skew-normal null** rather than a Gaussian null. It simulates data from that skew-normal and asks how often a two-Gaussian GMM obtains a BIC advantage at least as large as observed.

Finally, repeated 5-fold cross-validation evaluates held-out log predictive density. With 10 repeats, the diagnostic compares

$$
\frac1{N_{test}}
\sum \log f_{GMM}(z_{test})
-
\frac1{N_{test}}
\sum \log f_{SN}(z_{test}).
$$

Positive values favor the GMM predictively. No $p$-value is assigned to the repeated-CV difference because folds are dependent.


In [ ]:
RUN_HEAVY_MODELS = False

if RUN_HEAVY_MODELS:
    subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / "code" / "run_paper21_mixture.py"),
            "--materias", str(REPO_ROOT / "data" / "controlled" / "Materias_pseudonymized.csv"),
            "--asesorias", str(REPO_ROOT / "data" / "controlled" / "Asesorias_pseudonymized.csv"),
            "--gmm-bootstrap", "199",
            "--shape-bootstrap", "199",
            "--shape-cv-folds", "5",
            "--shape-cv-repeats", "10",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Using versioned, controlled-run mixture outputs. Set RUN_HEAVY_MODELS=True to recompute them.")


### 11.2 Reading and aligning the fitted GMM parameters

#### What this block does

This block reads the fitted two-component parameter tables for the numeric complete-case and imputed outcomes, then combines the historical 0–5 fits with separately estimated exact-6 and 7+ tail fits.

#### What is happening mathematically / technically?

For each fitted $K=2$ model, the library orders components by their estimated means,

$$
\hat\mu_L<\hat\mu_H,
$$

and reports

$$
(\hat\pi_L,\hat\mu_L,\hat\sigma_L),
\qquad
(\hat\pi_H,\hat\mu_H,\hat\sigma_H).
$$

The labels “lower-performance” and “higher-performance” are therefore **post-estimation ordering labels**. They solve the label-switching problem for reporting; they are not observed classes and they do not imply that a person has a fixed latent type.

The output also contains Ashman's separation statistic for $K=2$,

$$
D
=
\frac{\sqrt{2}\,|\hat\mu_H-\hat\mu_L|}
{\sqrt{\hat\sigma_L^2+\hat\sigma_H^2}},
$$

which summarizes separation relative to within-component spread.

Exact 6 and 7+ are re-estimated as their own groups rather than algebraically split from a previously fitted 6+ mixture. Concatenating the tables therefore aligns independently fitted density models into the reporting grid; it does not refit a joint mixture across attendance groups.


In [ ]:
# Combine historical 0-5/6+ outputs with the exact-6 and 7+ sensitivity outputs
# exactly as the manuscript does when discussing the high-frequency tail.
cc_params = pd.read_csv(TABLES_DIR / "38_complete_case_gmm_two_component_parameters.csv")
imp_params = pd.read_csv(TABLES_DIR / "42_imputed_gmm_two_component_parameters.csv")
cc_tail = pd.read_csv(TABLES_DIR / "54_sensitivity_6_7plus_complete_case_gmm_two_component_parameters.csv")
imp_tail = pd.read_csv(TABLES_DIR / "58_sensitivity_6_7plus_imputed_gmm_two_component_parameters.csv")

display(Markdown("### Numeric complete-case two-Gaussian parameters"))
display(pd.concat([
    cc_params.loc[cc_params["group"].astype(str).isin(["0","1","2","3","4","5"])],
    cc_tail,
], ignore_index=True))

display(Markdown("### Imputed-outcome two-Gaussian parameters"))
display(pd.concat([
    imp_params.loc[imp_params["group"].astype(str).isin(["0","1","2","3","4","5"])],
    imp_tail,
], ignore_index=True))


### 11.3 Distinguishing a finite mixture from ordinary skewness

#### What this block does

This block compares a one-Gaussian model, a one-component skew-normal model, and a two-Gaussian mixture on the same observations, separately for complete-case and imputed outcomes.

#### What is happening mathematically / technically?

A skew-normal has density

$$
f_{SN}(z;\xi,\omega,\alpha)
=
\frac{2}{\omega}
\phi\left(\frac{z-\xi}{\omega}\right)
\Phi\left(
\alpha\frac{z-\xi}{\omega}
\right),
$$

so it can represent substantial unimodal asymmetry with only one continuous component. This is the key alternative to interpreting every left-skewed histogram as a mixture.

The observed-sample comparison reports AIC and BIC for

$$
G_1,\qquad SN_1,\qquad GMM_2.
$$

For the direct skew-normal-vs-GMM check, the statistic is

$$
\Delta BIC
=
BIC(SN_1)-BIC(GMM_2),
$$

so $\Delta BIC>0$ favors the two-Gaussian mixture.

The parametric bootstrap fits $SN_1$, generates $B=199$ synthetic samples from that fitted null, refits both $SN_1$ and $GMM_2$ to every sample, and computes

$$
\hat p
=
\frac{
1+\sum_{b=1}^{B}
\mathbf 1(\Delta BIC_b\ge\Delta BIC_{obs})
}{
B+1
}.
$$

This directly asks whether the apparent GMM advantage is unusually large relative to data that are genuinely generated by one skewed component.

The repeated-CV diagnostic is orthogonal to BIC: it asks which fitted density assigns higher probability to unseen observations. Concordance among BIC, bootstrap, and held-out log density is much stronger evidence for distributional mixture structure than any one criterion alone.


In [ ]:
shape_cc = pd.read_csv(TABLES_DIR / "47_complete_case_skewnormal_vs_gmm.csv")
shape_imp = pd.read_csv(TABLES_DIR / "48_imputed_skewnormal_vs_gmm.csv")
shape_cc_tail = pd.read_csv(TABLES_DIR / "63_sensitivity_6_7plus_complete_case_skewnormal_vs_gmm.csv")
shape_imp_tail = pd.read_csv(TABLES_DIR / "64_sensitivity_6_7plus_imputed_skewnormal_vs_gmm.csv")

display(Markdown("### Shape-model comparison: observed numeric grades"))
display(shape_cc)
display(shape_cc_tail)

display(Markdown("### Shape-model comparison: imputed continuous outcome"))
display(shape_imp)
display(shape_imp_tail)


The observed numeric-grade result is deliberately conservative. A two-Gaussian model is clearly supported for some low-frequency groups, while several higher-frequency distributions are adequately represented by a single skew-normal. In particular, the apparent 7+ complete-case lower component is essentially an unstable extreme-tail fit rather than evidence for a stable second population.

The imputed outcome shows a clearer recurring lower component because administrative outcomes are numerically placed in the lower tail. This difference between observed and imputed specifications is itself scientifically informative: administrative course endings materially shape the distributional structure.


### 11.4 Visual diagnostics for the fitted mixture

#### What this block does

The first plot overlays the observed complete-case KDE with the fitted lower Gaussian, fitted higher Gaussian, and their weighted sum. The second compares the estimated lower-component weight across attendance frequencies for complete-case and imputed outcomes.

#### What is happening mathematically / technically?

Given the fitted parameters, the component curves are

$$
\hat f_L(z)
=
\hat\pi_L
\phi(z;\hat\mu_L,\hat\sigma_L^2),
\qquad
\hat f_H(z)
=
\hat\pi_H
\phi(z;\hat\mu_H,\hat\sigma_H^2),
$$

and the fitted mixture is

$$
\hat f(z)=\hat f_L(z)+\hat f_H(z).
$$

The empirical KDE and fitted GMM are shown on the same ridge only as a **model diagnostic**. A narrow component supported by one extreme observation can achieve a locally improved likelihood, so visual support, bootstrap behavior, and predictive validation are required before treating the second component as stable.

The second figure plots

$$
\hat\pi_{L,k}
$$

against attendance group $k$. Comparing $\hat\pi_{L,k}^{CC}$ with $\hat\pi_{L,k}^{IMP}$ isolates how much the numerical treatment of BV/RT/BA changes the amount of mass that EM allocates to the lower-density regime.


In [ ]:
# Complete-case Gaussian-mixture ridgeline and lower-component weight comparison.
render_paper_figure(paper21_figures.plot_complete_case_gmm_ridgeline)
render_paper_figure(paper21_figures.plot_lower_component_weight)


## 12. What is inside the lower imputed component?

The mixture is not converted into hard student labels. Instead, posterior component responsibilities are aggregated softly by the observed final state. This preserves uncertainty about component membership.

Among CMAT users, the lower imputed component is strongly enriched in administrative outcomes, especially BV, whereas the higher component is overwhelmingly PASS. This links the distributional result to the separate academic-management result without claiming that the lower component is a psychological student type.


### 12.1 Soft outcome-state composition using posterior responsibilities

#### What this block does

This block asks which observed academic-result states contribute probability mass to each fitted imputed-outcome component without assigning students deterministically to one component.

#### What is happening mathematically / technically?

After EM, each observation has posterior responsibilities

$$
r_{iL}=P(C_i=L\mid z_i),
\qquad
r_{iH}=P(C_i=H\mid z_i),
\qquad
r_{iL}+r_{iH}=1.
$$

A hard classifier would replace this vector with $\arg\max_k r_{ik}$, discarding uncertainty. The paper instead uses soft expected counts. For component $k$ and observed academic state $s$,

$$
\widehat N_{ks}
=
\sum_{i=1}^{n}
r_{ik}\mathbf 1(S_i=s),
$$

$$
\widehat N_k
=
\sum_i r_{ik},
\qquad
\widehat P(S=s\mid C=k)
=
\frac{\widehat N_{ks}}{\widehat N_k}.
$$

Therefore a student with $(r_{iL},r_{iH})=(0.60,0.40)$ contributes 0.60 expected observations to the lower component and 0.40 to the higher component. No row-level component label is required or saved.

This calculation describes the **composition implied by the fitted density model**. It does not convert the GMM into an observed latent-class outcome model.


In [ ]:
imputed_composition = pd.read_csv(TABLES_DIR / "44_imputed_gmm_soft_component_composition.csv")
imputed_tail_composition = pd.read_csv(TABLES_DIR / "60_sensitivity_6_7plus_imputed_gmm_soft_component_composition.csv")

display(imputed_composition)
display(imputed_tail_composition)


Aggregated across positive-attendance groups in the manuscript's validated analysis, the lower imputed component is approximately 55.5% administrative outcomes, including about 49.5% BV, whereas the higher component is approximately 97% PASS.

The appropriate interpretation is therefore:

> the completed outcome contains a lower distributional component strongly enriched in adverse administrative endings, especially BV.

The inappropriate interpretation would be that the model has discovered fixed categories such as “good students” and “bad students.”


## 13. Instructor-level clustering sensitivity

The primary specification clusters by instructor-period because that is also the fixed-effect grading context. As a dependence sensitivity, the paper re-clusters by instructor while retaining instructor-period fixed effects.

The broad zero-versus-positive benchmark remains nearly unchanged. Positive-frequency omnibus tests become somewhat more suggestive, but the result still does not become a clean adjacent visit-by-visit staircase.


### 13.1 Sensitivity to clustering at the instructor level

#### What this block does

This block reads the robustness results obtained by retaining the instructor-period fixed effects but changing the cluster identifier from instructor-period to instructor.

#### What is happening mathematically / technically?

The coefficient vector $\hat\beta$ is unchanged in principle because the regression design matrix and OLS objective are unchanged. What changes is the partition used in the sandwich “meat.”

Primary covariance:

$$
\widehat V_{\text{classroom}}
\propto
(X'X)^{-1}
\left[
\sum_{c}
X_c'\hat u_c\hat u_c'X_c
\right]
(X'X)^{-1}.
$$

Instructor sensitivity:

$$
\widehat V_{\text{prof}}
\propto
(X'X)^{-1}
\left[
\sum_{p}
X_p'\hat u_p\hat u_p'X_p
\right]
(X'X)^{-1},
$$

where $p$ pools all instructor-period observations belonging to the same instructor.

The second specification therefore allows arbitrary residual correlation across different academic periods taught by the same professor. It is a dependence-structure sensitivity, not a new confounding adjustment and not a different exposure estimand.


In [ ]:
instructor_benchmark = pd.read_csv(TABLES_DIR / "29_instructor_cluster_benchmark_0_vs_1plus.csv")
instructor_omnibus = pd.read_csv(TABLES_DIR / "28_instructor_cluster_omnibus.csv")

display(instructor_benchmark)
display(instructor_omnibus)


## 14. Discussion

The results support a layered interpretation of CMAT attendance.

First, the strongest and most stable empirical separation is between **zero attendance and any recorded attendance**. Every positive-frequency group lies above the zero group on adjusted standardized performance, and the broad benchmark is also large for PASS.

Second, frequency among users is not a smooth dose-response curve. The PPA-compatible first three visits are similar to one another, and the third visit does not coincide with an academic discontinuity. Later descriptive changes, including four-to-five and six-to-7+, are visible but imprecise.

Third, the 7+ group is a meaningful high-frequency profile because it has the highest observed mean and pass rate and because its adjusted PASS probability exceeds the one-visit group after multiplicity correction. This is evidence of a distinctive observed profile, not proof that the seventh visit itself causes the difference.

Fourth, adverse outcomes differ in **type**, not only in frequency. CMAT users who do not pass are relatively less likely to finish with a numeric failing grade and relatively more likely to have BV, while RT does not show the same adjusted contrast. This pattern is compatible with broader academic engagement or institutional navigation, but those mechanisms are not directly measured.

Finally, the distributional analysis shows why observed numeric grades and the imputed completed outcome must both be reported. Administrative outcomes materially sharpen the lower component of the imputed distribution. The stronger bimodality in the completed outcome therefore cannot be interpreted as a universal two-type structure among students.


## 15. Limits and identification

The principal limitation is self-selection. Students are not randomly assigned to CMAT, and visit frequency can be related to prior preparation, academic difficulty, motivation, schedule flexibility, persistence, advice seeking, and other unmeasured characteristics.

Timing is also endogenous. Students may attend because difficulties emerge, return because the service is useful, stop attending because they no longer need help, or have fewer opportunities to accumulate visits because they withdraw earlier.

PPA further complicates the behavioral meaning of the first three visits because the records do not identify which visits were specifically motivated by participation credit. Thus the third visit is not a valid regression-discontinuity cutoff or instrument in the present data.

The high-frequency cells are small, especially exact six and the exact counts above six before pooling, so uncertainty is visibly wider in the upper tail. Mixture components are density approximations rather than identified latent classes, and the numerical values assigned to BV/RT/BA are unobserved imputations whose role must remain transparent.

Consequently, the target estimand of the present notebook is descriptive/associational:

$$
E[Y\mid K=k, X] - E[Y\mid K=j, X],
$$

after the documented adjustment, rather than the causal potential-outcome contrast

$$
E[Y(k)-Y(j)].
$$


## 16. Conclusion

The full attendance profile contains substantially more structure than a simple user/non-user indicator, but it does not support a monotone dose-response interpretation.

The most robust result is the large **0 versus 1+** separation. Within positive attendance, the first one to three visits are difficult to interpret as purely voluntary because of PPA, and there is no observable break exactly at the third visit. The 7+ group is the clearest high-frequency profile, especially for PASS, while adjacent high-frequency contrasts remain imprecise.

The distributional extension adds an important second result: once administrative outcomes are incorporated into the completed continuous outcome, a lower performance component becomes much clearer and is strongly enriched in BV and other adverse endings. Observed numeric grades alone provide a more heterogeneous and less universally bimodal picture.

The defensible synthesis is therefore:

$$
\boxed{
\text{CMAT frequency is not a simple treatment dose;}
\quad
0\text{ vs }1+\text{ dominates,}
\quad
7+\text{ is distinctive,}
\quad
\text{and adverse-outcome composition matters.}
}
$$


## 17. Full canonical regeneration

All seven manuscript figures above are already **constructed and rendered inline** by the notebook. The cell below is separate: it is only for regenerating the repository's canonical aggregate CSV tables and saved vector-PDF artifacts from the controlled pseudonymized inputs.

It is intentionally not executed automatically because the full mixture bootstrap and repeated cross-validation are computationally expensive.


### 17.1 Full end-to-end regeneration

#### What this block does

When enabled, this block calls the repository build orchestration to regenerate the complete aggregate-table family and all publication figures from the controlled pseudonymized inputs.

#### What is happening mathematically / technically?

The dependency graph is

$$
\text{controlled row-level data}
\longrightarrow
\begin{cases}
\text{support audit}\\
\text{OLS/LPM/logit outcomes}\\
\text{GMM/skew-normal/bootstrap/CV}
\end{cases}
\longrightarrow
\text{aggregate CSVs}
\longrightarrow
\text{publication figures}.
$$

The point of this block is not to introduce another estimator. It verifies that the notebook's displayed results can be regenerated by the same canonical runners that produce the manuscript artifacts.

The expensive step is the repeated refitting of mixture models inside the parametric bootstraps and cross-validation folds. That is why the switch remains off during ordinary interactive reading.


In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    subprocess.run(
        [sys.executable, str(PAPER_DIR / "paper_build.py"), "--tables", "--figures"],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print(
        "Full pipeline not run. Set RUN_FULL_PIPELINE=True to regenerate all canonical "
        "aggregate tables and publication figures from the controlled pseudonymized inputs."
    )


## 18. Provenance and source files

The notebook is a presentation layer over the same analysis stack as the manuscript. For audit or modification, the canonical sources are:

- `paper/sections/01.tex`–`04.tex`: manuscript text;
- `paper/docs/analysis/ANALYSIS_PLAN.md`: analysis contract;
- `paper/docs/analysis/VISIT_GROUPING_DECISION.md`: outcome-blind grouping history;
- `paper/docs/results/PRELIMINARY_RESULTS.md`: verified result summary;
- `paper/docs/results/MIXTURE_ANALYSIS_RESULTS.md`: mixture-model validation and interpretation;
- `paper/docs/interpretation/ACADEMIC_MANAGEMENT_HYPOTHESIS.md`: mechanism guardrails;
- `code/run_paper21_outcomes.py`: main outcome analysis;
- `code/run_paper21_mixture.py`: numeric-failure and mixture analysis;
- `code/figures_paper21.py`: publication figures;
- `paper/references.bib`: bibliography.

The notebook should be updated when the canonical manuscript or analysis changes; scientific methods should continue to live in `cmat_analysis`, not be forked into notebook-only helper implementations.


### 18.1 Exact manuscript bibliography

#### What this block does

This final code block prints the BibTeX database used by the manuscript so the computational document and the paper share one reference source.

#### What is happening mathematically / technically?

There is no statistical operation here. The technical purpose is **single-source provenance**: the notebook reads `paper/references.bib` directly rather than maintaining a copied citation list. Consequently, changes to the manuscript bibliography are visible here without synchronizing a second bibliographic database.


In [ ]:
# Optional: show the exact bibliography used by the manuscript.
references = (PAPER_DIR / "references.bib").read_text(encoding="utf-8")
print(references)
